# Laden von OSM mittels osmium
- https://duckdb.org/community_extensions/extensions/osmium

In [ ]:
import duckdb
import os
import time
import requests

## Abfrage der aktuellen OSM-Daten (Standard nicht älter als 4 Tage)

In [ ]:
url = 'https://download.geofabrik.de/europe/germany/niedersachsen-latest.osm.pbf'
output_path = "downloads/niedersachsen-latest.osm.pbf"

In [ ]:
if os.path.exists(output_path):
    mtime = os.path.getmtime(output_path)
    age_hours = (time.time() - mtime) / 3600
    print(f"File '{output_path}' was last modified {age_hours:.1f} hours ago.")
else:
    print(f"File '{output_path}' does not exist.")

In [ ]:
# Download a pbf file to the downloads folder
# If the file is older than 96 hours, re-download it
if age_hours > 96:
    response = requests.get(url)
    with open(output_path, "wb") as f:
        f.write(response.content)
    print(f"Downloaded file to {output_path}")

## Start der Datenbank und Beispielabfragen

In [ ]:
duck = duckdb.connect()

In [ ]:
duck.sql("""
SELECT extension_name, installed, description
FROM duckdb_extensions();
""").df()

In [ ]:
duck.sql(""" install osmium from community;
load osmium;
install spatial;
load spatial;
SET geometry_always_xy = true;
-- INSTALL openfgdb FROM community;
-- LOAD openfgdb;

""")

In [ ]:
duck.sql("from st_drivers()").df()

In [ ]:
duck.sql(f"""

LOAD osmium;
SELECT id, tags['name'] AS name, geometry
FROM '{output_path}'
WHERE kind = 'node' AND tags['place'] = 'city';

""")

### Export der Wege nach Parquet/FGB

In [ ]:
duck.sql(f"""

LOAD osmium;

copy (

SELECT id, tags['name'] AS name, tags['highway'] AS highway, tags['foot'] AS foot, tags['level'] AS level, tags['layer'] AS layer,
tags['access'] AS access, tags['bus'] AS bus, tags['bus_allowed'] AS bus_allowed, tags['foot_allowed'] AS foot_allowed,
 geometry
FROM '{output_path}'
WHERE kind = 'line'
  AND tags['highway'] IS NOT NULL)

  to 'out/ways.geoparquet' (FORMAT 'parquet', COMPRESSION 'snappy')

""")

In [ ]:
duck.sql(f"""

LOAD osmium;

copy 
    (
    SELECT id, tags['name'] AS name, tags['highway'] AS highway, tags['foot'] AS foot, tags['level'] AS level, tags['layer'] AS layer,
    tags['access'] AS access, tags['bus'] AS bus, tags['bus_allowed'] AS bus_allowed, tags['foot_allowed'] AS foot_allowed,
    tags['oneway'] AS oneway, tags['maxspeed'] AS maxspeed, tags['lanes'] AS lanes, tags['surface'] AS surface,
    tags['bicycle'] AS bicycle, 
    st_transform(geometry, 'EPSG:4326', 'EPSG:25832') AS geometry
    FROM '{output_path}'
    WHERE kind = 'line'
        AND tags['highway'] IS NOT NULL
    )

 TO 'out/ways_na4.gdb' (FORMAT gdal, 
                        DRIVER 'OpenFileGDB', 
                        SRS 'EPSG:25832', 
                        LAYER_NAME 'ways_na', 
                        GEOMETRY_TYPE 'LINESTRING',
                                        LAYER_CREATION_OPTIONS (
                                                'FEATURE_DATASET=osm', 
                                                'LAYER_NAME=highways_nds_sel', 
                                                'LAYER_ALIAS=auswahl_osm'
                                                ));
""")